Spark has it's own core data source reading and writing capabilites 

- CSV
- JSON
- Parquet
- ORC
- JDBC /ODBC connection
- Plain-text files

Spark has numerous community-created data sources. Here’s just a small sample:
- Cassandra
- HBase
- MongoDB
- AWS Redshift
- XML
And many, many others

After we have a DataFrame reader, we specify several values:
- The format
- The schema
- The read mode
- A series of options

In [4]:
from pyspark.sql.session import SparkSession



spark = (SparkSession.builder.getOrCreate())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 14:20:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read modes

1. permissive:  *Sets all fields to null when it encounters a corrupted record and places all corrupted records in a string column called _corrupt_record*
2. dropMalformed: drop the row that contains malformed records, like schema is not working, or row is malformed
3. failFast: Faile immediately upon encoutering malformed records

### 1. permissive

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, StringType, IntegerType

df_schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),
        StructField("_corrupt_record", StringType(), True)

    ]
)

file_path = "../data/bad_record.csv"


df = (

        spark
        .read
        .format("csv")
        .option("header","true")
        .option("mode","permissive") # by default it's permissive
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(df_schema)
        .load(file_path)
)


df.show()


+----+-------+----+--------------------+
|  id|   name| age|     _corrupt_record|
+----+-------+----+--------------------+
|   1|  Alice|  30|                NULL|
|   2|    Bob|NULL|        2,Bob,twenty|
|   3|Charlie|  45|                NULL|
|NULL|   NULL|NULL|malformed_row_wit...|
|   4|  David|  50|                NULL|
+----+-------+----+--------------------+



## dropMalformed

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, StringType, IntegerType

df_schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),

    ]
)

file_path = "../data/bad_record.csv"


df = (

        spark
        .read
        .format("csv")
        .option("header","true")
        .option("mode","dropmalformed") # by default it's permissive
        .schema(df_schema)
        .load(file_path)
)


df.show()


+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  3|Charlie| 45|
|  4|  David| 50|
+---+-------+---+



## failFast

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, StringType, IntegerType
from pyspark.errors import AnalysisException, PySparkException, NumberFormatException

df_schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),

    ]
)

file_path = "../data/bad_record.csv"

# Suppress JVM ERROR stack traces in the console
spark.sparkContext.setLogLevel("FATAL")

try:
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("mode", "failfast")
        .schema(df_schema)
        .load(file_path)
    )
    df.show()
except NumberFormatException as e:
    print(f"Failed cleanly: {e.getErrorClass()}")
except PySparkException as e:
    print(f"Failed cleanly: {e.getErrorClass()}")
except Exception as e:
    print("Failed cleanly due to a bad record.")


Failed cleanly due to a bad record.


### Writing Data:

Instead of the
DataFrameReader, we have the DataFrameWriter.


dataFrame.write

### Spark Save Modes (`append`, `overwrite`, `errorIfExists`, `ignore`)

When writing data out of Spark to a destination, **Save modes** specify what will happen if Spark finds data or files already at the specified location. 

| Save Mode | Default? | Behavior |
| :--- | :--- | :--- |
| **`errorIfExists`** | **Yes (Default)** | Throws an error and fails the write if data or files already exist at the specified location. |
| **`append`** | No | Appends the output files to the list of files that already exist at that location. |
| **`overwrite`** | No | Will completely overwrite any data that already exists there. |
| **`ignore`** | No | If data or files exist at the location, do nothing with the current DataFrame. |

#### Important Nuances to Know:
1. **Default Behavior**: Because `errorIfExists` is the default, writing to a directory that already contains data will immediately fail your job to prevent accidental data loss.
2. **Immutability of Files**: Spark doesn't update individual rows in files. Modes like `append` simply add new files to the destination folder. `overwrite` deletes the target directory and writes completely new files.

In [35]:
import shutil
import os

data = [("James", "Sales", 3000), ("Maria", "Finance", 3000)]
columns = ["name", "department", "salary"]

df = spark.createDataFrame(data, schema=columns)

output_dir = "../data/tmp/spark_save_modes_test"

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

In [36]:
# 2. Initial Write (This will succeed because the directory doesn't exist yet)

(
    df
    .write
    .format("csv")
    .option("header","true")
    .save(output_dir)
)

print("count:",spark.read.csv(output_dir, header=True).count())

count: 2


In [37]:
# 1. errorIfExists Mode: Fails because the directory "../data/tmp/spark_save_modes_test" already exists

print("--- 1. errorIfExists Mode ---")

from pyspark.errors import AnalysisException

spark.sparkContext.setLogLevel("FATAL")

try:
    (
    df
    .write
    .format("csv")
    .option("header","true")
    .save(output_dir)
    )
except AnalysisException as e:
    print(f"Job failed as expected. Error:\n{str(e).split('Caused by:')[0]}")

--- 1. errorIfExists Mode ---
Job failed as expected. Error:
[PATH_ALREADY_EXISTS] Path file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/tmp/spark_save_modes_test already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04


In [38]:
# 2. ignore Mode: Silently skips the write without failing. The original data remains intact.
print("--- 2. ignore Mode ---")

(
df
.write
.format("csv")
.mode("ignore")
.option("header","true")
.save(output_dir)
)

print("count:",spark.read.csv(output_dir, header=True).count())

--- 2. ignore Mode ---
count: 2


In [39]:
# 3. append Mode: Adds new files to the existing directory without modifying the old ones.
print("--- 3. append Mode ---")
(
df
.write
.format("csv")
.mode("append")
.option("header","true")
.save(output_dir)
)

print("Appended successfully. Current row count:", spark.read.csv(output_dir, header=True).count())


--- 3. append Mode ---
Appended successfully. Current row count: 4


In [40]:
# 4. overwrite Mode: Deletes the existing directory/files entirely and writes the current DataFrame fresh.
print("--- 4. overwrite Mode ---")

# Overwriting with just the 'new_data_df' (Robert's row)
df.write \
  .format("csv") \
  .mode("overwrite") \
  .option("header","true") \
  .save(output_dir)

print("Overwritten successfully. The old rows are gone. Current row count:", spark.read.csv(output_dir, header=True).count())


--- 4. overwrite Mode ---
Overwritten successfully. The old rows are gone. Current row count: 2


In [1]:
spark.stop()

NameError: name 'spark' is not defined